# RECOMMENDATION SYSTEM TESTING/EXPERIMENTS

## Generate Test Data (may take a while to fully execute)

In [75]:
import os
import django

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()

import random
import numpy as np
import pandas as pd
from datetime import timedelta, date

from django.conf import settings
from django.contrib.auth import get_user_model
from django.contrib.auth.hashers import make_password

from apps.common.enums import WatchlistSource
from apps.movies.models import Movie, Genre
from apps.library.models import Rating, WatchlistEntry
from apps.recommendations.models import Swipe
from apps.recommendations.services.hybrid_pool import build_hybrid_pool

User = get_user_model()

N_USERS = 200
MIN_RATINGS_PER_USER = 15
MAX_RATINGS_PER_USER = 60
MIN_WATCHLIST_PER_USER = 0
MAX_WATCHLIST_PER_USER = 15
RATING_SCALE_MAX = getattr(settings, "RECOMMENDATION_RATING_SCALE_MAX", 5.0)

movies = list(Movie.objects.all())
n_movies = len(movies)
all_indices = np.arange(n_movies)
print(f"{n_movies} movies available")

# Zipf-style weights: some movies are "popular" (rated by many users),
# the rest are niche -> creates real overlap between users for CF
ranks = np.arange(1, n_movies + 1)
weights = 1 / ranks**0.8
weights /= weights.sum()

# 1) Bulk-create test users
new_users = [
    User(
        email=f"testuser{i}@example.com",
        username=f"testuser{i}",
        password=make_password("testpass123"),
        is_guest=False,
        is_active=True,
    )
    for i in range(N_USERS)
]
User.objects.bulk_create(new_users, ignore_conflicts=True)

test_users = list(User.objects.filter(username__startswith="testuser").order_by("username"))
print(f"{len(test_users)} test users ready")

# 2) Build ratings and watchlist entries in memory, then bulk_create
ratings_to_create = []
watchlist_to_create = []
today = date.today()

for user in test_users:
    # --- ratings: weighted (Zipf) sample over ALL movies ---
    n_rate = random.randint(MIN_RATINGS_PER_USER, MAX_RATINGS_PER_USER)
    rated_idx = np.random.choice(all_indices, size=min(n_rate, n_movies), replace=False, p=weights)
    rated_set = set(rated_idx)

    for idx in rated_idx:
        movie = movies[idx]
        value = round(random.triangular(1, RATING_SCALE_MAX, RATING_SCALE_MAX * 0.7), 1)
        ratings_to_create.append(Rating(
            user=user,
            movie=movie,
            title=movie.title,
            release_year=movie.release_year,
            rating=value,
            liked=value >= RATING_SCALE_MAX * 0.6,
            watched_date=today - timedelta(days=random.randint(1, 900)),
        ))

    # --- watchlist: weighted sample over movies not already rated ---
    remaining_idx = np.setdiff1d(all_indices, rated_idx, assume_unique=True)
    if remaining_idx.size:
        remaining_weights = weights[remaining_idx]
        remaining_weights = remaining_weights / remaining_weights.sum()

        n_watch = random.randint(MIN_WATCHLIST_PER_USER, MAX_WATCHLIST_PER_USER)
        watch_idx = np.random.choice(
            remaining_idx,
            size=min(n_watch, remaining_idx.size),
            replace=False,
            p=remaining_weights,
        )
        for idx in watch_idx:
            movie = movies[idx]
            watchlist_to_create.append(WatchlistEntry(
                user=user,
                movie=movie,
                title=movie.title,
                release_year=movie.release_year,
                added_date=today - timedelta(days=random.randint(1, 400)),
                source=WatchlistSource.IMPORTED,
            ))

Rating.objects.bulk_create(ratings_to_create, ignore_conflicts=True, batch_size=1000)
WatchlistEntry.objects.bulk_create(watchlist_to_create, ignore_conflicts=True, batch_size=1000)

print(f"{len(ratings_to_create)} ratings and {len(watchlist_to_create)} watchlist entries created")

1362 movies available
200 test users ready
7644 ratings and 1545 watchlist entries created


---------------------------------------------------------------------------------------------

In [76]:
def as_df(scored):
    return pd.DataFrame([
        {
            "title": c.movie.title,
            "year": c.movie.release_year,
            "content": round(c.content_score, 3),
            "cf": round(c.cf_score, 3) if c.cf_score is not None else None,
            "propagated": c.cf_was_propagated,
            "final": round(c.final_score, 3),
        }
        for c in scored
    ])

---------------------------------------------------------------------------------------------

## Random User Example

### Get Random User

In [85]:
user = (
    User.objects
    .filter(username__startswith="testuser", ratings__isnull=False)
    .distinct()
    .order_by("?")
    .first()
)

user_ratings = Rating.objects.filter(user=user).order_by("-rating")

print(f"\n===== {user.username} ({user_ratings.count()} ratings) =====")

print("--- rated movies ---")
display(pd.DataFrame([
    {"title": r.title, "year": r.release_year, "rating": r.rating, "liked": r.liked}
    for r in user_ratings
]))


===== testuser0 (33 ratings) =====
--- rated movies ---


,title,year,rating,liked
0,Good Will Hunting,1997,4.8,True
1,Project Hail Mary,2026,4.5,True
2,All About Eve,1950,4.3,True
3,Blade Runner 2049,2017,4.0,True
4,Toxic: A Fairy Tale for Grown-Ups,2026,3.9,True
5,Pinocchio: Unstrung,2026,3.9,True
6,Behind the Screen,1916,3.7,True
7,The Devil All the Time,2020,3.7,True
8,"Duck, You Sucker!",1971,3.7,True
9,Spider-Man: Homecoming,2017,3.7,True


### Content-Based Recommendations

In [86]:
print("--- Content-Based ---")
display(as_df(build_hybrid_pool(user, strategy="content")))

--- Content-Based ---


,title,year,content,cf,propagated,final
0,Dune,2021,0.863,None,False,0.863
1,Jurassic World Rebirth,2025,0.863,None,False,0.863
2,Doctor Strange in the Multiverse of Madness,2022,0.860,None,False,0.860
3,Avengers: Endgame,2019,0.859,None,False,0.859
4,Spider-Man: Across the Spider-Verse,2023,0.856,None,False,0.856
5,Arrival,2016,0.856,None,False,0.856
6,Guardians of the Galaxy Vol. 3,2023,0.856,None,False,0.856
7,The Batman,2022,0.856,None,False,0.856
8,Black Widow,2021,0.855,None,False,0.855
9,Jurassic World Dominion,2022,0.854,None,False,0.854


### Collaborative-Filtering-Based Recommendations

In [87]:
print("--- Collaborative-Filtering-Based ---")
display(as_df(build_hybrid_pool(user, strategy="collaborative")))

--- Collaborative-Filtering-Based ---


,title,year,content,cf,propagated,final
0,Black Widow,2021,0.855,0.920,False,0.920
1,Avengers: Age of Ultron,2015,0.853,0.682,True,0.682
2,Avengers: Endgame,2019,0.859,0.667,False,0.667
3,Thor: Ragnarok,2017,0.853,0.666,True,0.666
4,Suicide Squad,2016,0.850,0.630,True,0.630
5,Jurassic World Rebirth,2025,0.863,0.612,False,0.612
6,Avatar: The Way of Water,2022,0.854,0.606,True,0.606
7,Doctor Strange,2016,0.853,0.604,True,0.604
8,The Amazing Spider-Man 2,2014,0.849,0.602,True,0.602
9,Guardians of the Galaxy Vol. 3,2023,0.856,0.599,False,0.599


### Hybrid Approach (Content + Collaborative)

#### Hybrid Appraoch with Alpha = 0.2 (more weight to collaborative)

In [88]:
print("--- Hybrid (alpha=0.2) ---")
display(as_df(build_hybrid_pool(user, strategy="hybrid", alpha=0.2)))

--- Hybrid (alpha=0.2) ---


,title,year,content,cf,propagated,final
0,Black Widow,2021,0.855,0.920,False,0.907
1,Avengers: Age of Ultron,2015,0.853,0.682,True,0.716
2,Avengers: Endgame,2019,0.859,0.667,False,0.705
3,Thor: Ragnarok,2017,0.853,0.666,True,0.704
4,Suicide Squad,2016,0.850,0.630,True,0.674
5,Jurassic World Rebirth,2025,0.863,0.612,False,0.662
6,Avatar: The Way of Water,2022,0.854,0.606,True,0.656
7,Doctor Strange,2016,0.853,0.604,True,0.654
8,The Amazing Spider-Man 2,2014,0.849,0.602,True,0.652
9,Guardians of the Galaxy Vol. 3,2023,0.856,0.599,False,0.650


#### Hybrid Appraoch with Alpha = 0.5 (same weight to both appraoches)

In [89]:
print("--- Hybrid (alpha=0.5) ---")
display(as_df(build_hybrid_pool(user, strategy="hybrid", alpha=0.5)))

--- Hybrid (alpha=0.5) ---


,title,year,content,cf,propagated,final
0,Black Widow,2021,0.855,0.920,False,0.887
1,Avengers: Age of Ultron,2015,0.853,0.682,True,0.767
2,Avengers: Endgame,2019,0.859,0.667,False,0.763
3,Thor: Ragnarok,2017,0.853,0.666,True,0.760
4,Suicide Squad,2016,0.850,0.630,True,0.740
5,Jurassic World Rebirth,2025,0.863,0.612,False,0.737
6,Avatar: The Way of Water,2022,0.854,0.606,True,0.730
7,Doctor Strange,2016,0.853,0.604,True,0.728
8,Guardians of the Galaxy Vol. 3,2023,0.856,0.599,False,0.727
9,The Amazing Spider-Man 2,2014,0.849,0.602,True,0.726


#### Hybrid Appraoch with Alpha = 0.8 (more weight to content)

In [90]:
print("--- Hybrid (alpha=0.8) ---")
display(as_df(build_hybrid_pool(user, strategy="hybrid", alpha=0.8)))

--- Hybrid (alpha=0.8) ---


,title,year,content,cf,propagated,final
0,Black Widow,2021,0.855,0.920,False,0.868
1,Avengers: Endgame,2019,0.859,0.667,False,0.821
2,Avengers: Age of Ultron,2015,0.853,0.682,True,0.819
3,Thor: Ragnarok,2017,0.853,0.666,True,0.816
4,Jurassic World Rebirth,2025,0.863,0.612,False,0.813
5,Suicide Squad,2016,0.850,0.630,True,0.806
6,Guardians of the Galaxy Vol. 3,2023,0.856,0.599,False,0.804
7,Avatar: The Way of Water,2022,0.854,0.606,True,0.804
8,Doctor Strange,2016,0.853,0.604,True,0.803
9,Doctor Strange in the Multiverse of Madness,2022,0.860,0.574,True,0.803


---------------------------------------------------------------------------------------------

## Delete Test Data

In [ ]:
# CAUTION: THIS WILL DELETE EVERY USER WHOSE USERNAME STARTS WITH 'testuser'

deleted_count, details = User.objects.filter(username__startswith="testuser").delete()
print(f"{deleted_count} rows deleted")
print(details)